In [ ]:
# Modelo de Previsão de Churn - TelecomPlus
# Autor: Ryan Gartlan

# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. CARREGAMENTO E EXPLORAÇÃO DE DADOS

# Carregando os dados
print("Carregando os dados...")
dados_treino = pd.read_csv('dados_clientes.csv')
dados_teste = pd.read_csv('desafio.csv')

# Informações básicas
print(f"\nDimensões do conjunto de treino: {dados_treino.shape}")
print(f"Dimensões do conjunto de teste: {dados_teste.shape}")

# Análise da variável alvo (churn)
print("\nDistribuição de churn:")
print(dados_treino['churn'].value_counts())
print(f"Taxa de churn: {dados_treino['churn'].mean()*100:.2f}%")

# Amostragem para análise rápida (30% dos dados originais)
# Isso acelera a análise exploratória sem perder padrões importantes
amostra_treino = dados_treino.sample(frac=0.3, random_state=42)
print(f"\nUsando uma amostra de {amostra_treino.shape[0]} registros para análise exploratória rápida")

# Análise de correlação com churn (variáveis numéricas)
colunas_numericas = ['idade', 'tempo_como_cliente', 'suporte_contatado', 'chamados_abertos', 
                   'tempo_medio_atendimento', 'reclamacoes', 'atrasos_pagamento', 
                   'servicos_assinados', 'valor_mensal', 'total_gasto']

corr_numericas = amostra_treino[colunas_numericas].corrwith(amostra_treino['churn'])
print("\nCorrelação das variáveis numéricas com churn:")
print(corr_numericas.sort_values(ascending=False))

# Análise de variáveis categóricas
colunas_categoricas = ['genero', 'estado_civil', 'tipo_contrato', 'forma_pagamento', 'renda_faixa']

print("\nDistribuição das variáveis categóricas em relação ao churn:")
for coluna in colunas_categoricas:
    print(f"\n{coluna.upper()}:")
    print(pd.crosstab(amostra_treino[coluna], amostra_treino['churn'], normalize='index'))

# Visualizações principais (limitadas para economia de tempo)
plt.figure(figsize=(15, 5))

# 1. Churn por tipo de contrato (frequentemente a feature mais importante)
plt.subplot(1, 3, 1)
sns.countplot(x='tipo_contrato', hue='churn', data=amostra_treino)
plt.title('Churn por Tipo de Contrato')
plt.legend(title='Churn', loc='upper right')

# 2. Churn por tempo como cliente
plt.subplot(1, 3, 2)
sns.boxplot(x='churn', y='tempo_como_cliente', data=amostra_treino)
plt.title('Tempo como Cliente vs Churn')

# 3. Churn por número de reclamações
plt.subplot(1, 3, 3)
sns.countplot(x='reclamacoes', hue='churn', data=amostra_treino)
plt.title('Churn por Número de Reclamações')
plt.legend(title='Churn', loc='upper right')

plt.tight_layout()
plt.savefig('analise_churn.png')
plt.close()

In [ ]:
# 2. PRÉ-PROCESSAMENTO DOS DADOS

# Tratamento da coluna produtos_assinados
print("\nTratando a coluna 'produtos_assinados'...")

def extrair_produtos(produto_str):
    """Extrai produtos da string e retorna uma lista"""
    if pd.isna(produto_str) or produto_str == '[]':
        return []
    
    # Método mais simples e rápido para extrair produtos
    produtos = []
    if "Produto A" in produto_str:
        produtos.append("Produto A")
    if "Produto B" in produto_str:
        produtos.append("Produto B")
    if "Produto C" in produto_str:
        produtos.append("Produto C")
    if "Produto D" in produto_str:
        produtos.append("Produto D")
    if "Produto E" in produto_str:
        produtos.append("Produto E")
    if "Produto F" in produto_str:
        produtos.append("Produto F")
    
    return produtos

# Aplicando a função para extrair produtos
dados_treino['produtos_lista'] = dados_treino['produtos_assinados'].apply(extrair_produtos)
dados_teste['produtos_lista'] = dados_teste['produtos_assinados'].apply(extrair_produtos)

# Criando variáveis dummy para cada produto
produtos = ["Produto A", "Produto B", "Produto C", "Produto D", "Produto E", "Produto F"]
for produto in produtos:
    col_name = f"tem_{produto.replace(' ', '_')}"
    dados_treino[col_name] = dados_treino['produtos_lista'].apply(lambda x: 1 if produto in x else 0)
    dados_teste[col_name] = dados_teste['produtos_lista'].apply(lambda x: 1 if produto in x else 0)

# Criando features adicionais (engenharia de features)
print("\nCriando features adicionais...")

# 1. Gasto médio mensal
dados_treino['gasto_medio_mensal'] = dados_treino['total_gasto'] / dados_treino['tempo_como_cliente'].replace(0, 1)
dados_teste['gasto_medio_mensal'] = dados_teste['total_gasto'] / dados_teste['tempo_como_cliente'].replace(0, 1)

# 2. Relação entre reclamações e tempo como cliente
dados_treino['taxa_reclamacoes'] = dados_treino['reclamacoes'] / dados_treino['tempo_como_cliente'].replace(0, 1)
dados_teste['taxa_reclamacoes'] = dados_teste['reclamacoes'] / dados_teste['tempo_como_cliente'].replace(0, 1)

# 3. Relação entre atrasos e tempo como cliente
dados_treino['taxa_atrasos'] = dados_treino['atrasos_pagamento'] / dados_treino['tempo_como_cliente'].replace(0, 1)
dados_teste['taxa_atrasos'] = dados_teste['atrasos_pagamento'] / dados_teste['tempo_como_cliente'].replace(0, 1)

# Removendo colunas que não serão usadas no modelo
colunas_para_remover = ['id_cliente', 'produtos_assinados', 'produtos_lista']
X_treino = dados_treino.drop(colunas_para_remover + ['churn'], axis=1)
y_treino = dados_treino['churn']
X_teste = dados_teste.drop(colunas_para_remover, axis=1)

# Identificando colunas numéricas e categóricas para o pipeline de pré-processamento
colunas_numericas = X_treino.select_dtypes(include=['int64', 'float64']).columns.tolist()
colunas_categoricas = X_treino.select_dtypes(include=['object']).columns.tolist()

print(f"\nFeatures numéricas ({len(colunas_numericas)}): {colunas_numericas}")
print(f"Features categóricas ({len(colunas_categoricas)}): {colunas_categoricas}")

# Pipeline de pré-processamento
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, colunas_numericas),
        ('cat', categorical_transformer, colunas_categoricas)
    ])

# Divisão dos dados para validação
X_train, X_val, y_train, y_val = train_test_split(
    X_treino, y_treino, test_size=0.2, random_state=42, stratify=y_treino
)

print(f"\nConjunto de treino: {X_train.shape[0]} registros")
print(f"Conjunto de validação: {X_val.shape[0]} registros")

In [ ]:
# 3. INSIGHTS E HIPÓTESES

print("\nCom base na análise exploratória, podemos formular as seguintes hipóteses:")

print("\n1. Clientes com contratos mensais têm maior probabilidade de churn")
print("   - A flexibilidade do contrato mensal facilita o cancelamento")

print("\n2. O tempo como cliente influencia negativamente o churn")
print("   - Clientes mais antigos tendem a ser mais fiéis")

print("\n3. Clientes com mais reclamações têm maior probabilidade de cancelamento")
print("   - Insatisfação com o serviço é um forte indicador de churn")

print("\n4. Clientes com maiores valores mensais podem ter maior tendência a cancelar")
print("   - Sensibilidade a preço é um fator importante")

print("\n5. Número de serviços assinados pode reduzir a probabilidade de churn")
print("   - Maior integração com a empresa dificulta a saída")

In [ ]:
# 4. TREINAMENTO E COMPARAÇÃO DE MODELOS

# Implementando 3 modelos diferentes (requisito mínimo: pelo menos 2)
print("\nTreinando modelos de classificação...")

# Modelo 1: Regressão Logística
modelo_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])

print("\nTreinando Regressão Logística...")
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_val)
y_prob_lr = modelo_lr.predict_proba(X_val)[:, 1]

acc_lr = accuracy_score(y_val, y_pred_lr)
prec_lr = precision_score(y_val, y_pred_lr)
rec_lr = recall_score(y_val, y_pred_lr)
f1_lr = f1_score(y_val, y_pred_lr)
auc_lr = roc_auc_score(y_val, y_prob_lr)

print(f"Acurácia: {acc_lr:.4f}")
print(f"Precisão: {prec_lr:.4f}")
print(f"Recall: {rec_lr:.4f}")
print(f"F1-Score: {f1_lr:.4f}")
print(f"AUC-ROC: {auc_lr:.4f}")

# Modelo 2: Random Forest (reduzido para execução mais rápida)
modelo_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced'))
])

print("\nTreinando Random Forest...")
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_val)
y_prob_rf = modelo_rf.predict_proba(X_val)[:, 1]

acc_rf = accuracy_score(y_val, y_pred_rf)
prec_rf = precision_score(y_val, y_pred_rf)
rec_rf = recall_score(y_val, y_pred_rf)
f1_rf = f1_score(y_val, y_pred_rf)
auc_rf = roc_auc_score(y_val, y_prob_rf)

print(f"Acurácia: {acc_rf:.4f}")
print(f"Precisão: {prec_rf:.4f}")
print(f"Recall: {rec_rf:.4f}")
print(f"F1-Score: {f1_rf:.4f}")
print(f"AUC-ROC: {auc_rf:.4f}")

# Modelo 3: Gradient Boosting
modelo_gb = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42, n_estimators=100))
])

print("\nTreinando Gradient Boosting...")
modelo_gb.fit(X_train, y_train)
y_pred_gb = modelo_gb.predict(X_val)
y_prob_gb = modelo_gb.predict_proba(X_val)[:, 1]

acc_gb = accuracy_score(y_val, y_pred_gb)
prec_gb = precision_score(y_val, y_pred_gb)
rec_gb = recall_score(y_val, y_pred_gb)
f1_gb = f1_score(y_val, y_pred_gb)
auc_gb = roc_auc_score(y_val, y_prob_gb)

print(f"Acurácia: {acc_gb:.4f}")
print(f"Precisão: {prec_gb:.4f}")
print(f"Recall: {rec_gb:.4f}")
print(f"F1-Score: {f1_gb:.4f}")
print(f"AUC-ROC: {auc_gb:.4f}")

# Resumo dos resultados
resultados = {
    'Regressão Logística': {
        'Acurácia': acc_lr,
        'Precisão': prec_lr,
        'Recall': rec_lr,
        'F1-Score': f1_lr,
        'AUC-ROC': auc_lr,
        'Modelo': modelo_lr
    },
    'Random Forest': {
        'Acurácia': acc_rf,
        'Precisão': prec_rf,
        'Recall': rec_rf,
        'F1-Score': f1_rf,
        'AUC-ROC': auc_rf,
        'Modelo': modelo_rf
    },
    'Gradient Boosting': {
        'Acurácia': acc_gb,
        'Precisão': prec_gb,
        'Recall': rec_gb,
        'F1-Score': f1_gb,
        'AUC-ROC': auc_gb,
        'Modelo': modelo_gb
    }
}

# Criando DataFrame para visualização
resultados_df = pd.DataFrame({
    modelo: {metrica: valor for metrica, valor in metricas.items() if metrica != 'Modelo'}
    for modelo, metricas in resultados.items()
}).T

print("\nResumo comparativo dos modelos:")
print(resultados_df)

# Identificando o melhor modelo com base na acurácia
melhor_modelo_nome = resultados_df['Acurácia'].idxmax()
melhor_modelo = resultados[melhor_modelo_nome]['Modelo']

print(f"\nMelhor modelo: {melhor_modelo_nome}")
print(f"Acurácia: {resultados_df.loc[melhor_modelo_nome, 'Acurácia']:.4f}")

In [ ]:
# 5. OTIMIZAÇÃO DO MELHOR MODELO (GRID SEARCH SIMPLIFICADO)

# Parâmetros específicos para cada tipo de modelo
if melhor_modelo_nome == 'Regressão Logística':
    param_grid = {
        'classifier__C': [0.1, 1.0, 10.0],
        'classifier__penalty': ['l2']
    }
elif melhor_modelo_nome == 'Random Forest':
    param_grid = {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [10, 20, None],
        'classifier__min_samples_split': [2, 5]
    }
else:  # Gradient Boosting
    param_grid = {
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [3, 5]
    }

print(f"\nOtimizando o modelo {melhor_modelo_nome}...")

# Grid Search com número reduzido de combinações para maior velocidade
grid_search = GridSearchCV(
    melhor_modelo, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)

print("\nIniciando busca de hiperparâmetros...")
grid_search.fit(X_train, y_train)

print("\nMelhores hiperparâmetros encontrados:")
print(grid_search.best_params_)

# Modelo otimizado
modelo_otimizado = grid_search.best_estimator_

# Avaliação do modelo otimizado
y_pred_otimizado = modelo_otimizado.predict(X_val)
y_prob_otimizado = modelo_otimizado.predict_proba(X_val)[:, 1]

acc_otimizado = accuracy_score(y_val, y_pred_otimizado)
auc_otimizado = roc_auc_score(y_val, y_prob_otimizado)

print(f"\nDesempenho do modelo otimizado:")
print(f"Acurácia: {acc_otimizado:.4f}")
print(f"AUC-ROC: {auc_otimizado:.4f}")

print("\nMatriz de confusão:")
conf_matrix = confusion_matrix(y_val, y_pred_otimizado)
print(conf_matrix)

print("\nRelatório de classificação:")
print(classification_report(y_val, y_pred_otimizado))

In [ ]:
# 6. ANÁLISE DE FEATURE IMPORTANCE

# Para obter feature importance, precisamos aplicar o preprocessador e obter os nomes das colunas
X_train_processed = preprocessor.fit_transform(X_train)

# Se for Random Forest ou Gradient Boosting, podemos obter importance diretamente
if melhor_modelo_nome in ['Random Forest', 'Gradient Boosting']:
    print("\nFeature importance do modelo:")
    
    feature_importances = modelo_otimizado.named_steps['classifier'].feature_importances_
    
    # Obtendo nomes das features após transformação
    feature_names = []
    
    # Nomes das colunas após one-hot encoding
    if len(colunas_categoricas) > 0:
        cat_encoder = modelo_otimizado.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
        feature_names.extend(cat_encoder.get_feature_names_out(colunas_categoricas).tolist())
    
    # Adicionando colunas numéricas
    feature_names.extend(colunas_numericas)
    
    # Criando DataFrame de importância
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': feature_importances
    }).sort_values('Importance', ascending=False)
    
    # Mostrando as 15 features mais importantes
    print("\nTop 15 features mais importantes:")
    print(importance_df.head(15))
    
    # Visualização
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df.head(15))
    plt.title(f'Top 15 Features Mais Importantes - {melhor_modelo_nome}')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()
    
else:  # Regressão Logística
    print("\nCoeficientes da Regressão Logística:")
    
    coefs = modelo_otimizado.named_steps['classifier'].coef_[0]
    
    # Obtendo nomes das features após transformação
    feature_names = []
    
    # Nomes das colunas após one-hot encoding
    if len(colunas_categoricas) > 0:
        cat_encoder = modelo_otimizado.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
        feature_names.extend(cat_encoder.get_feature_names_out(colunas_categoricas).tolist())
    
    # Adicionando colunas numéricas
    feature_names.extend(colunas_numericas)
    
    # Criando DataFrame de coeficientes
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefs
    })
    
    # Ordenando por valor absoluto dos coeficientes
    coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()
    coef_df = coef_df.sort_values('Abs_Coef', ascending=False)
    
    # Mostrando os 15 coeficientes mais importantes
    print("\nTop 15 features mais importantes (por magnitude de coeficiente):")
    print(coef_df[['Feature', 'Coefficient']].head(15))
    
    # Visualização
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Coefficient', y='Feature', data=coef_df.head(15))
    plt.title('Top 15 Coeficientes - Regressão Logística')
    plt.tight_layout()
    plt.savefig('coeficientes_logistica.png')
    plt.close()

In [ ]:
# 7. TREINAMENTO DO MODELO FINAL E GERAÇÃO DE PREVISÕES
print("\n" + "=" * 80)
print("7. TREINAMENTO DO MODELO FINAL E GERAÇÃO DE PREVISÕES")
print("=" * 80)

# Treinamento do modelo final com todos os dados
print("\nTreinando o modelo final com todos os dados...")
modelo_final = modelo_otimizado
modelo_final.fit(X_treino, y_treino)

# Previsões no conjunto de teste
print("\nGerando previsões para o conjunto de teste...")
y_pred_teste = modelo_final.predict(X_teste)

# Criação do arquivo de resultados
resultado = pd.DataFrame({
    'id_cliente': dados_teste['id_cliente'],
    'churn': y_pred_teste.astype(int)
})

# Exibindo as primeiras linhas do resultado
print("\nPrimeiras 5 linhas do resultado:")
print(resultado.head())

# Estatísticas das previsões
print(f"\nTotal de clientes no conjunto de teste: {resultado.shape[0]}")
print(f"Taxa de churn prevista: {resultado['churn'].mean()*100:.2f}%")

# Salvando o arquivo de resultados
resultado.to_csv('resultado_ryan_gartlan.csv', index=False)
print("\nArquivo 'resultado_ryan_gartlan.csv' gerado com sucesso!")

In [ ]:
# 8. CONCLUSÕES

print("\nPrincipais insights do projeto:")

print("\n1. O modelo", melhor_modelo_nome, "teve o melhor desempenho, com acurácia de", 
      f"{acc_otimizado:.4f} após otimização.")

print("\n2. Os fatores mais importantes para prever churn são:")
if melhor_modelo_nome in ['Random Forest', 'Gradient Boosting']:
    for i, row in importance_df.head(5).iterrows():
        print(f"   - {row['Feature']}: {row['Importance']:.4f}")
else:
    for i, row in coef_df.head(5).iterrows():
        print(f"   - {row['Feature']}: {row['Coefficient']:.4f}")

print("\n3. Recomendações para redução de churn:")
print("   - Incentivar a migração para contratos anuais")
print("   - Criar programas de fidelidade para clientes mais antigos")
print("   - Melhorar o atendimento para reduzir reclamações")
print("   - Oferecer pacotes com múltiplos serviços")